# Phase 1 — Data Loading, Schema Mapping & Cleaning

**Business question this notebook supports:** *How large is the problem, and what's the revenue at risk?* (Section 1, sub-question 1) — and indirectly every later phase, since this notebook produces the single order-grain table (`data/processed/orders_clean.parquet`) that Phases 2-5 all build on.

Two parts:
1. **Schema mapping** — load every raw table, confirm shape/dtypes, document how the tables join and each table's grain.
2. **Cleaning** — filter to a well-defined analysis population (delivered orders) and roll everything up to one row per order, with every transformation decision justified before it's applied.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

# Jupyter kernels default to the notebook's own folder as cwd, not the repo
# root the rest of this notebook assumes for every relative path (per
# CLAUDE.md). Force it explicitly rather than relying on how the notebook
# happens to have been launched.
if Path.cwd().name == "notebooks":
    os.chdir("..")

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.data_loader import load_raw_tables, verify_raw_files_present
from src import cleaning

pd.set_option("display.max_columns", 50)

## Verify raw data is present

Per the project brief, Phase 1 stops here if any expected raw file is missing rather than guessing or proceeding with partial data.

In [2]:
missing = verify_raw_files_present()
if missing:
    raise FileNotFoundError(f"Missing expected raw files: {missing}")
print("All 9 expected raw files present in data/raw/.")

All 9 expected raw files present in data/raw/.


In [3]:
tables = load_raw_tables()
for name, df in tables.items():
    print(f"{name:22s} {df.shape[0]:>9,} rows x {df.shape[1]} cols")

orders                    99,441 rows x 8 cols
order_items              112,650 rows x 7 cols
order_reviews             99,224 rows x 7 cols
order_payments           103,886 rows x 5 cols
customers                 99,441 rows x 5 cols
sellers                    3,095 rows x 4 cols
products                  32,951 rows x 9 cols
geolocation            1,000,163 rows x 5 cols
category_translation          71 rows x 2 cols


In [4]:
for name, df in tables.items():
    print(f"\n=== {name} dtypes ===")
    print(df.dtypes)


=== orders dtypes ===
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

=== order_items dtypes ===
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

=== order_reviews dtypes ===
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       date

## Schema Mapping

The dataset is a normalized, order-centric schema spread across 9 CSVs. `orders` is the hub table: every other table joins back to it either directly (`order_id`) or transitively through `order_items` (`product_id`, `seller_id`) or `customers` (`customer_id`).

```mermaid
erDiagram
    ORDERS ||--o{ ORDER_ITEMS : "order_id"
    ORDERS ||--o{ ORDER_PAYMENTS : "order_id"
    ORDERS ||--o{ ORDER_REVIEWS : "order_id"
    ORDERS }o--|| CUSTOMERS : "customer_id"
    ORDER_ITEMS }o--|| PRODUCTS : "product_id"
    ORDER_ITEMS }o--|| SELLERS : "seller_id"
    PRODUCTS }o--|| CATEGORY_TRANSLATION : "product_category_name"
    CUSTOMERS ||--o{ GEOLOCATION : "zip_code_prefix (optional)"
    SELLERS ||--o{ GEOLOCATION : "zip_code_prefix (optional)"
```

### Grain and gotchas, table by table

**`orders`** (99,441 rows) — one row per order. `order_status` breaks down as 96,478 delivered (97.0%), with the remainder split across shipped, canceled, unavailable, invoiced, processing, created and approved. Gotcha: `order_delivered_customer_date` is null for most non-delivered rows, and even for 8 rows that *are* marked delivered — a data inconsistency handled explicitly below, not silently.

**`order_items`** — 112,650 rows, one row per line item within an order (`order_item_id` disambiguates items within the same order). Only 98,666 distinct `order_id`s appear here — orders in early-stage statuses (e.g. `created`) may not have generated item rows yet. Gotcha: an order can span multiple sellers, so seller-level joins need a `distinct_seller_count`, not a naive one-to-one assumption.

**`order_reviews`** — 99,224 rows. Gotcha: only ~98,673 distinct `order_id`s appear — 551 review rows are duplicates of an `order_id` (resubmitted or re-sent survey). Must dedupe by `order_id` (keep most recent `review_answer_timestamp`) before joining at order grain.

**`order_payments`** — 103,886 rows, one row per payment method / installment plan used per order. 2,961 orders split payment across more than one row (e.g. part boleto + part voucher). Aggregating to order level requires summing `payment_value` and choosing one representative `payment_type`.

**`customers`** — 99,441 rows, one per `customer_id` — but `customer_id` is minted fresh per order, not per person. Only 96,096 distinct `customer_unique_id` values exist. Gotcha: any repeat-customer analysis (Phase 2's 90-day repeat-purchase cohort) must use `customer_unique_id`, never `customer_id`.

**`sellers`** — 3,095 rows, one per `seller_id`, with zip/city/state.

**`products`** — 32,951 rows, one per `product_id`. Gotcha: 610 products (1.9%) have a null `product_category_name` (and correspondingly null name/description length and photo count) — bucketed into an `'unknown'` category below rather than dropped, since the product and any orders containing it are still valid records.

**`product_category_name_translation`** — 71 rows, Portuguese → English category name. Gotcha: 2 category names present in `products` (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) have no translation row; per the brief's fallback rule these keep their Portuguese name instead of being dropped or nulled.

**`geolocation`** (optional) — 1,000,163 rows, many rows per `zip_code_prefix` (multiple lat/lng samples). Not used in Phase 1 cleaning; `customer_state`/`seller_state` already on the customers/sellers tables cover the state-level analysis Phase 2-4 need.

## Cleaning

Every step below states the decision and the *why* before the code that performs it. None of the steps in this notebook drop more than 3% of orders — see the summary table at the end.

### Filter to `order_status == 'delivered'`

Review scores measure the post-delivery fulfillment experience. Orders that were canceled, are still in transit, or were marked unavailable never completed that journey, so including them would blend "never happened" outcomes into a metric that's specifically about delivery experience. We filter to delivered orders only — this is the population definition specified in the project brief, not a new judgment call.

In [5]:
delivered = cleaning.filter_delivered(tables["orders"])
n_raw = len(tables["orders"])
n_delivered = len(delivered)
print(f"orders: {n_raw:,} raw -> {n_delivered:,} delivered "
      f"({n_delivered / n_raw:.1%} retained, {1 - n_delivered / n_raw:.1%} dropped)")
tables["orders"]["order_status"].value_counts()

orders: 99,441 raw -> 96,478 delivered (97.0% retained, 3.0% dropped)


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Parse timestamps and compute `delivery_gap_days`

All five order timestamp columns are already parsed to datetime in `load_raw_tables`. `delivery_gap_days = order_delivered_customer_date - order_estimated_delivery_date`, in days; positive means the order arrived after the estimate (late), negative means early.

8 delivered orders (0.01%) have a null `order_delivered_customer_date` despite being marked delivered — a data inconsistency, not a business signal, and there's no gap to compute without an actual delivery date. 8 rows is far below the 5% check-in threshold, so we drop them here rather than fabricating a date.

In [6]:
delivered = cleaning.compute_delivery_gap(delivered)
n_missing_delivery_date = delivered["delivery_gap_days"].isna().sum()
print(f"delivered orders missing actual delivery date: {n_missing_delivery_date} "
      f"({n_missing_delivery_date / len(delivered):.3%})")

delivered = delivered.dropna(subset=["delivery_gap_days"]).copy()
delivered["delivery_gap_days"] = delivered["delivery_gap_days"].astype(int)
print(f"orders remaining after dropping missing-delivery-date rows: {len(delivered):,}")

delivered orders missing actual delivery date: 8 (0.008%)
orders remaining after dropping missing-delivery-date rows: 96,470


### Map product categories to English, with a fallback

`products.product_category_name` is in Portuguese. We left-join `product_category_name_translation` to get the English name, falling back to the original Portuguese name when a category has no translation row, and to the literal string `'unknown'` when `product_category_name` itself is null. No product rows are dropped by this step.

In [7]:
products_mapped = cleaning.map_product_categories(tables["products"], tables["category_translation"])

unmapped = (
    set(tables["products"]["product_category_name"].dropna().unique())
    - set(tables["category_translation"]["product_category_name"])
)
print("categories with no translation row (fall back to Portuguese):", unmapped)
print("products with null category name (fall back to 'unknown'):",
      tables["products"]["product_category_name"].isna().sum())

categories with no translation row (fall back to Portuguese): {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}
products with null category name (fall back to 'unknown'): 610


### Aggregate `order_items` to order grain

Each order can contain multiple line items across multiple sellers. We collapse to one row per `order_id`: `total_price` and `total_freight` (summed across items), `item_count` (row count), and `distinct_seller_count` (nunique sellers, flags split-seller orders).

We also assign each order a single `primary_category`: the category of its highest-price line item. Multi-category orders exist but are uncommon, and using the highest-revenue item's category is a standard convention that keeps the order-grain table at exactly one row per order for the category-level analysis Phases 2-3 need (seller-in-category ranking, category heatmap).

In [8]:
items_agg = cleaning.aggregate_order_items(tables["order_items"])
primary_category = cleaning.primary_category_per_order(tables["order_items"], products_mapped)

print(f"item aggregation covers {len(items_agg):,} orders")
items_agg.head()

item aggregation covers 98,666 orders


,order_id,total_price,total_freight,item_count,distinct_seller_count
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1


### Aggregate `order_payments` to order grain

2,961 orders split their payment across more than one row (part boleto + part voucher, or installments logged as separate sequential rows). We sum `payment_value` into `total_payment_value`, take the max `payment_installments` as `max_installments`, and take the `payment_type` of the single highest-value row as the order's primary method — defining "primary" by where most of the money went, rather than a row-count mode.

In [9]:
payments_agg = cleaning.aggregate_order_payments(tables["order_payments"])
print(f"payment aggregation covers {len(payments_agg):,} orders")
payments_agg["payment_type"].value_counts()

payment aggregation covers 99,440 orders


payment_type
credit_card    74975
boleto         19784
voucher         3151
debit_card      1527
not_defined        3
Name: count, dtype: int64

### Deduplicate `order_reviews`

Review rows whose `order_id` already has another review row (resubmitted or re-sent survey) need to collapse to one row per order before any order-grain join. We keep the most recent by `review_answer_timestamp` and drop the rest.

In [10]:
reviews_deduped, n_review_dupes_dropped = cleaning.dedupe_reviews(tables["order_reviews"])
print(f"review rows dropped as duplicates: {n_review_dupes_dropped} "
      f"({n_review_dupes_dropped / len(tables['order_reviews']):.2%} of review rows)")
print(f"reviews remaining after dedup: {len(reviews_deduped):,}")

review rows dropped as duplicates: 551 (0.56% of review rows)
reviews remaining after dedup: 98,673


### Outlier treatment on `price` and `freight_value`

We use the standard 1.5x IQR rule as a *flag*, not an automatic filter — the brief is explicit that extreme values need a manual sanity check before anything is dropped, because a high freight charge for a heavy/bulky item (furniture, appliances) is a legitimate cost, not a data error.

In [11]:
price_outlier_flag = cleaning.flag_iqr_outliers(tables["order_items"], "price")
freight_outlier_flag = cleaning.flag_iqr_outliers(tables["order_items"], "freight_value")

n_items = len(tables["order_items"])
print(f"price: {price_outlier_flag.sum():,} / {n_items:,} rows flagged ({price_outlier_flag.mean():.1%})")
print(f"freight_value: {freight_outlier_flag.sum():,} / {n_items:,} rows flagged ({freight_outlier_flag.mean():.1%})")

print("\nrows with price <= 0:", (tables["order_items"]["price"] <= 0).sum())
print("rows with freight_value < 0:", (tables["order_items"]["freight_value"] < 0).sum())

flagged = tables["order_items"][price_outlier_flag | freight_outlier_flag]
flagged.sort_values("price", ascending=False)[["order_id", "product_id", "price", "freight_value"]].head(10)

price: 8,427 / 112,650 rows flagged (7.5%)
freight_value: 11,613 / 112,650 rows flagged (10.3%)

rows with price <= 0: 0
rows with freight_value < 0: 0


,order_id,product_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,3999.90,17.01


**Decision: 0 rows dropped.** Flag rates are high (double digits on freight) because IQR is a tight rule on a right-skewed price distribution, not because the data is broken. Spot-checking the most extreme rows: `price` is never <= 0, `freight_value` is never negative, and the highest freight charges consistently pair with high item prices (e.g. price=979.00 with freight=409.68) — exactly the "heavy/bulky item, not a typo" pattern the brief calls out as legitimate. We retain every row; the flags are documented here, not applied as a filter.

### Assemble the order-grain table

Left-joining every piece above onto the delivered-orders base, so an order with no review yet (or missing a payment/item aggregation) still gets a row — review-score-specific analyses in later phases should filter on non-null `review_score` explicitly, rather than that filtering happening silently here.

In [12]:
customers_slim = tables["customers"][["customer_id", "customer_unique_id", "customer_city", "customer_state"]]

orders_clean = (
    delivered
    .merge(customers_slim, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(primary_category, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(
        reviews_deduped[["order_id", "review_score", "review_answer_timestamp"]],
        on="order_id",
        how="left",
    )
)

print(f"orders_clean shape: {orders_clean.shape}")
orders_clean.head()

orders_clean shape: (96470, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_gap_days,customer_unique_id,customer_city,customer_state,total_price,total_freight,item_count,distinct_seller_count,primary_category,total_payment_value,max_installments,payment_type,review_score,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,29.99,8.72,1,1,housewares,38.71,1.0,voucher,4.0,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6,af07308b275d755c9edb36a90c618231,barreiras,BA,118.70,22.76,1,1,perfumery,141.46,1.0,boleto,4.0,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,159.90,19.22,1,1,auto,179.12,3.0,credit_card,5.0,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,45.00,27.20,1,1,pet_shop,72.20,1.0,credit_card,5.0,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,19.90,8.72,1,1,stationery,28.62,1.0,credit_card,5.0,2018-02-18 13:02:51


### Sanity checks before saving

Row count should equal the post-filter delivered-order count (the joins are all left joins onto that base, so they can't add or lose rows). Key structural columns should have no unexpected nulls; `review_score` nulls are expected (orders without a submitted review) and must be handled explicitly downstream, not silently.

In [13]:
assert orders_clean["order_id"].is_unique, "orders_clean should be one row per order"
assert len(orders_clean) == len(delivered), "row count drifted during the joins"

key_cols = ["order_id", "customer_id", "delivery_gap_days", "total_price", "total_freight", "item_count"]
print("null counts in key columns:")
print(orders_clean[key_cols].isna().sum())

print(f"\norders missing a review_score: {orders_clean['review_score'].isna().sum():,} "
      f"({orders_clean['review_score'].isna().mean():.2%})")
print(f"orders missing payment aggregation: {orders_clean['total_payment_value'].isna().sum():,}")

orders_clean["review_score"].value_counts(dropna=False).sort_index()

null counts in key columns:
order_id             0
customer_id          0
delivery_gap_days    0
total_price          0
total_freight        0
item_count           0
dtype: int64

orders missing a review_score: 646 (0.67%)
orders missing payment aggregation: 1


review_score
1.0     9351
2.0     2921
3.0     7916
4.0    18888
5.0    56748
NaN      646
Name: count, dtype: int64

### Save cleaned table

In [14]:
OUTPUT_PATH = Path("data/processed/orders_clean.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
orders_clean.to_parquet(OUTPUT_PATH, index=False)
print(f"saved {len(orders_clean):,} rows x {orders_clean.shape[1]} cols to {OUTPUT_PATH}")

saved 96,470 rows x 22 cols to data\processed\orders_clean.parquet


### Item-grain table for seller- and product-level analysis (Phase 2)

`orders_clean` is deliberately order-grain, so it collapses multi-seller and multi-product orders down to `distinct_seller_count` and a single `primary_category` — it has no `seller_id` column at all. Phase 2's seller-ranking queries (Q4, Q6) and its product-weight-adjusted delivery query (Q8) need one row per order line item instead, carrying `seller_id` and `product_weight_g`.

We rebuild from `tables["order_items"]`, restrict to the same delivered-order population as `orders_clean` (so the two tables always agree on which orders are in scope), attach each item's category and weight from `products_mapped`, and pull `order_purchase_timestamp`, `customer_state`, and `review_score` straight from `orders_clean` rather than re-deriving them. Rows with a missing or non-positive `product_weight_g` are dropped — a shipped physical product can't weigh zero or nothing, so this is a data-quality drop, not a judgment call, and it affects a negligible share of rows (checked below, well under the 5% threshold).

In [15]:
items_with_attrs = cleaning.merge_item_product_attrs(tables["order_items"], products_mapped)
items_with_attrs = items_with_attrs[items_with_attrs["order_id"].isin(orders_clean["order_id"])].copy()

invalid_weight = cleaning.flag_invalid_weight(items_with_attrs)
print(f"items dropped for missing/non-positive product_weight_g: {invalid_weight.sum():,} / "
      f"{len(items_with_attrs):,} ({invalid_weight.mean():.3%})")
items_with_attrs = items_with_attrs[~invalid_weight].copy()

order_items_clean = (
    items_with_attrs
    .rename(columns={"category_final": "category"})[
        ["order_id", "seller_id", "product_id", "category", "product_weight_g"]
    ]
    .merge(
        orders_clean[[
            "order_id", "order_purchase_timestamp", "order_delivered_customer_date",
            "customer_state", "review_score",
        ]],
        on="order_id",
        how="inner",
    )
)
order_items_clean["actual_delivery_days"] = (
    order_items_clean["order_delivered_customer_date"] - order_items_clean["order_purchase_timestamp"]
).dt.days
order_items_clean = order_items_clean.drop(columns=["order_delivered_customer_date"])

print(f"order_items_clean shape: {order_items_clean.shape}")
print(f"distinct orders covered: {order_items_clean['order_id'].nunique():,} / {len(orders_clean):,} "
      f"({order_items_clean['order_id'].nunique() / len(orders_clean):.2%})")
print(f"distinct sellers covered: {order_items_clean['seller_id'].nunique():,}")
order_items_clean.head()

items dropped for missing/non-positive product_weight_g: 26 / 110,189 (0.024%)
order_items_clean shape: (110163, 9)
distinct orders covered: 96,448 / 96,470 (99.98%)
distinct sellers covered: 2,970


,order_id,seller_id,product_id,category,product_weight_g,order_purchase_timestamp,customer_state,review_score,actual_delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,650.0,2017-09-13 08:59:02,RJ,5.0,7
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,pet_shop,30000.0,2017-04-26 10:53:06,SP,4.0,16
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,3050.0,2018-01-14 14:33:31,MG,5.0,7
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc,perfumery,200.0,2018-08-08 10:00:35,SP,4.0,6
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089,garden_tools,3750.0,2017-02-04 13:57:51,SP,5.0,25


In [16]:
assert order_items_clean["seller_id"].notna().all(), "every order_items row has a seller_id"
assert order_items_clean["category"].notna().all(), "category_final fallback should leave no nulls"

print("null counts in order_items_clean:")
print(order_items_clean.isna().sum())

null counts in order_items_clean:
order_id                      0
seller_id                     0
product_id                    0
category                      0
product_weight_g              0
order_purchase_timestamp      0
customer_state                0
review_score                827
actual_delivery_days          0
dtype: int64


In [17]:
ITEM_OUTPUT_PATH = Path("data/processed/order_items_clean.parquet")
order_items_clean.to_parquet(ITEM_OUTPUT_PATH, index=False)
print(f"saved {len(order_items_clean):,} rows x {order_items_clean.shape[1]} cols to {ITEM_OUTPUT_PATH}")

saved 110,163 rows x 9 cols to data\processed\order_items_clean.parquet


## Summary

| Step | Rows in | Rows out | % dropped |
|---|---|---|---|
| Filter to `delivered` | 99,441 | 96,478 | 2.98% |
| Drop missing actual-delivery-date | 96,478 | 96,470 | 0.01% |
| **Total retained from raw orders** | 99,441 | 96,470 | **97.0%** |

No step in this notebook drops more than 3% of orders, well under the 5% check-in threshold, and no business-framing decision (delivered-only population, review-score scale) was changed from what the README specifies.

Two outputs are ready for Phase 2:
- `data/processed/orders_clean.parquet` — order-grain, for Q0, Q1, Q2, Q3, Q7.
- `data/processed/order_items_clean.parquet` — item-grain (one row per line item, carrying `seller_id` and `product_weight_g`), for Q4, Q6, Q8.